# Informed Search

**The central question:** If several moves are possible, which one deserves our attention next?

**Content:**
- Refresher
- Heuristics
  - admissability
  - consistency
- Algorithms
  - greedy best-first search
  - beam search
  - hill climbing
  - A* and proof for its optimality
  - IDA*
  - simulated annealing
- Homework problem
  - Problem statement
  - Heuristic choice

In [ ]:
from heapq import heappush, heappop
from itertools import count
from math import inf, exp
import random

## Refresher
- b, m, d
- Admissible, optimal, time complexity, space complexity

## Heuristic
A function which accepts the current state and returns an estimate for the distance from the current state to the goal state

If we have a delivery vehicle which should reach the goal location in an optimal way
| Symbol | Meaning | Delivery example |
|:--               |:--                           |:--|
| $g(n)$           | Cost paid to reach state $n$ | Distance already driven |
| $h(n)$           | Estimated remaining cost | Straight-line distance to the destination |
| $f(n)=g(n)+h(n)$             | Estimated total route cost | Distance driven plus distance still expected |

## Greedy best-first search

> Choose the frontier state with the smallest heuristic estimate $h$.

The goal is to find the path which leads to the solution

- **Complete**: No, but Complete in finite space with repeated-state checking
- **Optimal** : No
- **Time complexity** : $O(b^m)$ - a misleading "bad" heuristic may lead us to possibly explore all nodes
- **Space complexity** : $O(b^m)$ - keep all nodes in memory
- **Data structure**: priority queue based entirely on the heuristic

In [ ]:
def greedy_search(graph, start, goal, h):
    order = count()
    frontier = [(h[start], next(order), start)]
    parents = {start: None}

    while frontier:
        _, _, node = heappop(frontier)

        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parents[node]
            return path[::-1]

        for neighbor, _ in graph[node]:
            if neighbor not in parents:
                parents[neighbor] = node
                heappush(frontier, (h[neighbor], next(order), neighbor))

    return None

## Beam search

> Expand one level, rank its candidates, and keep only the best $l$.

- Expand: having the $l$ nodes expand all their children (max is $l*b$ nodes)
- Pruning: preserve only the $l$ with the lowest heuristic. (We need to sort out at most $l*b$ nodes, typically this is $O(log(lb)lb)$ , but also a more efficient sorting algorithm can be used and thus we simplify it to $O(lb)$)

- **Complete**: No, local search
- **Optimal** : No
- **Time complexity** : $O(bml)$
- **Space complexity**: $O(bl)$ or $O(bml)$ if each beam path is preserved

In [ ]:
def beam_search(graph, start, goal, h, width=2):
    beam = [[start]]
    visited = {start}

    while beam:
        candidates = []
        for path in beam:
            current = path[-1]
            if current == goal:
                  return path

            for neighbor, _ in graph[current]:
                if neighbor not in visited:
                    candidates.append(path + [neighbor])

        beam = sorted(candidates, key=lambda path: h[path[-1]])[:width]
        visited.update(path[-1] for path in beam)

    return None

## Hill climbing

> Move to the best neighbor only if it improves the current score.

- Used for optimization: higher scores mean better states.
- Stops at a local maximum or a plateau, even if a better state exists elsewhere.
- **Random restarts** can help escape local traps, but do not guarantee success in a finite number of runs.

- **Complete**: No
- **Optimal**: No, may stop at a local maximum
- **Time complexity**: $O(bm)$ for at most $m$ improving moves
- **Space complexity**: $O(b)$ when storing the generated neighbors
- **Data structure**: current state and its neighbors; no frontier

In [ ]:
def hill_climb(neighbors, score, start):
    current = start

    while neighbors[current]:
        candidate = max(neighbors[current], key=score.get)

        if score[candidate] <= score[current]:  # Local optimum
            break

        current = candidate

    return current

## Admissibility

> A heuristic never overestimates the cheapest remaining cost.

$$0 \leq h(n) \leq h^*(n)$$

- $h^*(n)$ is the true cheapest cost from $n$ to a goal; $h(G)=0$.
- Example: straight-line distance cannot exceed the shortest road distance.
- To prove admissibility, show that $h$ solves an easier problem obtained by removing constraints.

## Consistency

> The heuristic cannot decrease by more than the cost of a move.

$$h(n) \leq c(n,n') + h(n')$$

- Check this inequality for every edge $n \rightarrow n'$.
- With $h(G)=0$, consistency implies admissibility; the reverse is not always true.
- A consistent heuristic makes $f=g+h$ nondecreasing along a path.

In [ ]:
def is_consistent(graph, h):
    return all(h[node] <= cost + h[neighbor]
               for node in graph
               for neighbor, cost in graph[node])

## A* search

> Choose the frontier state with the smallest $f(n)=g(n)+h(n)$.

- $g(n)$: cost already paid; $h(n)$: estimated remaining cost.
- Return when the goal is removed from the priority queue, not when it is first discovered.
- Keep the cheapest known cost for each state. Reopen it if a cheaper route appears.

- **Complete**: Yes, with finite branching and positive step costs bounded away from zero
- **Optimal**: Yes, with an admissible heuristic and reopening; consistency avoids re-expansion
- **Time complexity**: $O\!\left(b^{\,1+\lfloor C^*/\varepsilon\rfloor}\right)$, can be similar to UCS
  - Often summarized as $O(b^d)$ or $O(b^(d+1))$ (keep in mind that an optimal path to the optimal solution can have 3 nodes while a suboptimal path can have 10 nodes and the queue could expand 9 of them)
- **Space complexity**: Same as the time complexity
- **Data structure**: priority queue based on $g+h$

The queue stores states. Parent links reconstruct the route after reaching the goal.

In [ ]:
def a_star(graph, start, goal, h):
    order = count()
    # f, counter, cost_from_start, node
    frontier = [(h[start], next(order), 0, start)]
    best_g = {start: 0}
    parents = {start: None}

    while frontier:
        _, _, cost_from_start, node = heappop(frontier)
        if cost_from_start != best_g[node]:
            continue  # Skip an outdated queue entry.

        if node == goal:
            path = []
            while node is not None:
                path.append(node)
                node = parents[node]
            return path[::-1]

        for neighbor, weight in graph[node]:
            new_cost = cost_from_start + weight
            if new_cost < best_g.get(neighbor, inf):
                best_g[neighbor] = new_cost
                parents[neighbor] = node
                heappush(frontier, (new_cost + h[neighbor], next(order),
                                    new_cost, neighbor))

    return None

## Why A* is optimal

Let $C^*$ be the optimal solution cost and $G_2$ a more expensive goal.
Assume that the heuristic is consistent

- Before finding an optimal goal, the queue contains a state $n$ on an optimal path with $f(n) \leq C^*$.
- Imagine we are expanding the $G_2$. Since $h(G_2)=0$, $f(G_2)=g(G_2)>C^*$
- This leads to a contradiction because $n$ should be in the queue and it should be chosen before the $G_2$


## A* optimality in a tree
In a tree, we need the heurustic to be only admissible in order to make the algorithm optimal because there is only one way to reach a node.

With a bad admissible heuristic we could very early add a suboptimal solution.
The suboptimal solution cost in the queue will still be its actual cost and it will be removed after the optimal solution.

## Reopening a state

> An admissible but inconsistent heuristic may require expanding the same state again.

```text
S ──1──▶ A ──1──▶ B ──2──▶ G
└────────3───────▶ B
h(S)=4, h(A)=3, h(B)=0, h(G)=0
```

- B is first expanded with $g=3$; G is discovered with cost 5.
- A then provides a cheaper route to B with $g=2$.
- Reopening B gives the optimal route S → A → B → G, costing 4.
- Permanently closing B would lose this improvement. Consistency prevents this situation.

## Memory Bounded A*
A* can lead to a high memory consumption $O(b^d)$. There are different ways to limit this amount
- A* with a limited priority queue (SMLA\*): uses the same priority queue but with limited capacity, prining the removed nodes and backing up them with the value of the best forgotten child
- Recursive Best First Search (RBFS)
- Iterative deepening A* (we are going to see it)

## Iterative deepening A*

> Repeat depth-first search with an increasing $f=g+h$ threshold.

- Start with threshold $h(start)$.
- Prune a path when its $f$ exceeds the threshold.
- If no goal is found, use the smallest exceeded $f$ as the next threshold.
- Check cycles on the active path; states can be revisited in later passes.

- **Complete**: Yes, with finite branching and positive step costs bounded away from zero
- **Optimal**: Yes, with an admissible heuristic and $h(G)=0$
- **Time complexity**: exponential; repeated thresholds can add substantial work
- **Space complexity**: $O(d)$ with lazy neighbor iteration, or $O(bd)$ if successors are stored at each depth
- **Data structure**: recursion stack and current path

## Simulated annealing

> Sometimes accept a worse neighbor to escape a local maximum.

- Pick a random neighbor and calculate $\Delta=score(candidate)-score(current)$.
- Always accept improvements. Accept other moves with probability $e^{\Delta/T}$.
- Reduce temperature $T$ after each step: worse moves become less likely.
- Return the best state encountered, which may differ from the final current state.

- **Complete**: No guarantee for a finite run
- **Optimal**: No guarantee for a finite run
- **Time complexity**: $O(t)$ for $t$ proposals with constant-time sampling and scoring
- **Space complexity**: $O(1)$ extra memory
- **Data structure**: current state, best state, and temperature

## Summary

| Algorithm | Next choice | Main limitation |
|:--|:--|:--|
| Greedy best-first | Lowest $h$ | Ignores cost already paid |
| Beam search | Best $l$ candidates at the next level | Can discard the only successful branch |
| Hill climbing | Best improving neighbor | Stops at local maxima or plateaus |
| A* | Lowest $g+h$ | Stores a potentially large frontier |
| IDA* | Depth-first search within an $f$ threshold | Repeats work across passes |
| Simulated annealing | Random neighbor with temperature-based acceptance | Results depend on sampling and cooling |

## Homework problem

> Solve the eight-puzzle using A* and compare two heuristics.

- **State**: a tuple of nine values; `0` represents the blank.
- **Action**: swap the blank with a tile directly above, below, left, or right.
- **Step cost**: 1 per move.
- **Goal**: `(1, 2, 3, 4, 5, 6, 7, 8, 0)`.
- Use the start state below. Return the solution path and its number of moves.

```text
Start          Goal
1 2 3          1 2 3
4 0 6          4 5 6
7 5 8          7 8 0
```

In [ ]:
start_board = (1, 2, 3, 4, 0, 6, 7, 5, 8)
goal_board = (1, 2, 3, 4, 5, 6, 7, 8, 0)

## Heuristic choice

- **Misplaced tiles**: count numbered tiles outside their goal positions.
- **Manhattan distance**: sum each numbered tile's horizontal and vertical distance to its goal.

$$h_M(n)=\sum_{tile\ne0} |row-row_{goal}|+|col-col_{goal}|$$

- Ignore the blank in both heuristics. Both are admissible and consistent for unit-cost moves.
- Manhattan distance dominates misplaced tiles: every misplaced tile needs at least one move.
- A legal move changes one tile's Manhattan distance by exactly 1, proving consistency.
- The maximum of admissible heuristics is admissible; their sum need not be.

## Homework tasks

- Implement `successors(board)` returning legal `(next_board, 1)` moves.
- Adapt A* to generate successors and evaluate `h(board)` on demand instead of reading a predefined graph and heuristic dictionary.
- Run A* with each heuristic. Compare solution cost and the number of expanded states.
- Explain why Manhattan distance is a lower bound and why it may guide search better.
- Optional: solve a harder solvable board with IDA* and compare memory use.